# 00_project_setup_and_preregistration

This notebook defines the fixed analysis protocol for the equine thermography Explainable-AI study.

It intentionally remains a standalone notebook. Later notebooks read the JSON/CSV outputs saved here.

**Key fixed assumption for the current project:** the clinical class comes from the folder structure:

```text
train/healthy, train/injured
valid/healthy, valid/injured
test/healthy, test/injured
```

Expert XML files are used for hotspot localization only, not as the primary source of the healthy/injured classification.

## 1. Imports and paths

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, re, warnings
from datetime import datetime, timezone
import pandas as pd
import numpy as np


BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, MODEL_SELECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project paths initialized")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DATA_DIR:", SPLIT_DATA_DIR)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Project paths initialized
PROJECT_ROOT: /content/project_thermography_equine
SPLIT_DATA_DIR: /content/project_thermography_equine/data/dataset_split
ANNOTATIONS_DIR: /content/project_thermography_equine/data/annotations
OUTPUT_ROOT: /content/project_thermography_equine/outputs


## 2. Reproducibility

In [2]:
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print(f"Seed set to {SEED}")
print(f"Torch available: {TORCH_AVAILABLE}")

Seed set to 42
Torch available: True


## 3. Fixed study protocol

In [3]:
study_protocol = {
    "study_title": "Explainable AI for thermographic detection of equine musculoskeletal pathology and hotspot localization",
    "analysis_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "unit_of_analysis": "one thermographic image corresponding to one unique horse",
    "one_image_one_horse_assumption": True,
    "classification_source_of_truth": "folder_structure",
    "folder_label_mapping": {"healthy": "healthy", "injured": "pathological"},
    "positive_class": "pathological",
    "negative_class": "healthy",
    "primary_endpoint": "image-level discrimination of pathological vs healthy horses",
    "primary_metric": "ROC AUC on the independent test set",
    "secondary_metrics": ["sensitivity", "specificity", "F1", "balanced_accuracy", "PPV", "NPV"],
    "localization_endpoints": ["IoU between saliency/hotspot localization and expert bounding boxes", "pointing game hit rate"],
    "annotation_source": "CVAT XML files from two independent experts",
    "annotation_role": "localization ground truth for hotspot analyses; not the primary clinical label source",
    "healthy_rule": "Images in healthy folders are treated as clinically healthy even if they have no XML objects. Empty XML image entries mean no expert-marked hotspot, not a missing clinical label.",
    "conflict_rule": "Images placed in healthy folders but containing expert hotspot annotations are flagged in annotation_label_conflicts.csv for manual review and are not silently relabeled.",
    "split_policy": "predefined train/valid/test split supplied by folders; no re-splitting in modeling notebooks",
    "reporting_guidelines": ["STARD 2015", "TRIPOD+AI", "CLAIM", "ARRIVE 2.0 as supporting animal-study reporting guidance"],
    "image_extensions": [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"],
    "random_seed": SEED
}

analysis_config = {
    "project_name": PROJECT_NAME,
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "split_data_dir": str(SPLIT_DATA_DIR),
    "metadata_dir": str(METADATA_DIR),
    "annotations_dir": str(ANNOTATIONS_DIR),
    "output_root": str(OUTPUT_ROOT),
    "config_dir": str(CONFIG_DIR),
    "reports_dir": str(REPORTS_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "models_dir": str(MODELS_DIR),
    "model_selection_dir": str(MODEL_SELECTION_DIR),
    "splits": ["train", "valid", "test"],
    "class_folders": {"healthy": 0, "injured": 1},
    "class_names": {"healthy": "healthy", "injured": "pathological"},
    "id_column": "horse_id",
    "image_name_column": "image_name",
    "label_column": "label_clinical",
    "binary_label_column": "label_binary",
    "positive_class": "pathological",
    "negative_class": "healthy",
    "primary_metric": "roc_auc",
    "random_seed": SEED
}

print(json.dumps(study_protocol, indent=2)[:1500])

{
  "study_title": "Explainable AI for thermographic detection of equine musculoskeletal pathology and hotspot localization",
  "analysis_timestamp_utc": "2026-06-06T15:16:26.117323+00:00",
  "unit_of_analysis": "one thermographic image corresponding to one unique horse",
  "one_image_one_horse_assumption": true,
  "classification_source_of_truth": "folder_structure",
  "folder_label_mapping": {
    "healthy": "healthy",
    "injured": "pathological"
  },
  "positive_class": "pathological",
  "negative_class": "healthy",
  "primary_endpoint": "image-level discrimination of pathological vs healthy horses",
  "primary_metric": "ROC AUC on the independent test set",
  "secondary_metrics": [
    "sensitivity",
    "specificity",
    "F1",
    "balanced_accuracy",
    "PPV",
    "NPV"
  ],
  "localization_endpoints": [
    "IoU between saliency/hotspot localization and expert bounding boxes",
    "pointing game hit rate"
  ],
  "annotation_source": "CVAT XML files from two independent exper

## 4. Expected annotation files

In [4]:
expected_annotation_files = {
    "train": {"expert1": "annotations_train_expert1.xml", "expert2": "annotations_train_expert2.xml"},
    "valid": {"expert1": "annotations_valid_expert1.xml", "expert2": "annotations_valid_expert2.xml"},
    "test":  {"expert1": "annotations_test_expert1.xml",  "expert2": "annotations_test_expert2.xml"},
}

annotation_config = {
    "expected_annotation_files": expected_annotation_files,
    "supported_alternative_patterns": [
        "annotations_{split}_expert{expert}.xml",
        "annotations_{split}_expert{expert}(*).xml"
    ],
    "hotspot_bbox_label": "hotspot_bbox",
    "hotspot_point_label": "hotspot_point",
    "no_hotspot_label": "no_hotspot"
}

print(pd.DataFrame([
    {"split": s, "expert": e, "expected_file": f}
    for s, ex in expected_annotation_files.items()
    for e, f in ex.items()
]))

   split   expert                  expected_file
0  train  expert1  annotations_train_expert1.xml
1  train  expert2  annotations_train_expert2.xml
2  valid  expert1  annotations_valid_expert1.xml
3  valid  expert2  annotations_valid_expert2.xml
4   test  expert1   annotations_test_expert1.xml
5   test  expert2   annotations_test_expert2.xml


## 5. Save protocol/config outputs

In [5]:
for path, obj in [
    (CONFIG_DIR / "study_protocol.json", study_protocol),
    (CONFIG_DIR / "analysis_config.json", analysis_config),
    (CONFIG_DIR / "annotation_config.json", annotation_config),
]:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print("Saved:", path)

protocol_summary = pd.DataFrame([
    {"field": k, "value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v}
    for k, v in study_protocol.items()
])
protocol_summary.to_csv(REPORTS_DIR / "study_protocol_summary.csv", index=False)
protocol_summary.to_csv(CONFIG_DIR / "study_protocol_summary.csv", index=False)
print("Saved protocol summary")

Saved: /content/project_thermography_equine/outputs/config/study_protocol.json
Saved: /content/project_thermography_equine/outputs/config/analysis_config.json
Saved: /content/project_thermography_equine/outputs/config/annotation_config.json
Saved protocol summary


## 6. Environment record

In [6]:
import platform, sys
try:
    from importlib.metadata import version, PackageNotFoundError
except Exception:
    from importlib_metadata import version, PackageNotFoundError

package_names = ["numpy", "pandas", "scikit-learn", "opencv-python", "Pillow", "matplotlib", "torch", "torchvision"]
packages = {}
for name in package_names:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = "not_installed"
    except Exception as e:
        packages[name] = f"version_check_failed: {e}"

environment_record = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": packages,
    "recorded_utc": datetime.now(timezone.utc).isoformat()
}
with open(CONFIG_DIR / "environment_record.json", "w", encoding="utf-8") as f:
    json.dump(environment_record, f, indent=2)

pd.DataFrame([{"package": k, "version": v} for k, v in packages.items()]).to_csv(REPORTS_DIR / "environment_packages.csv", index=False)
print(json.dumps(environment_record, indent=2))

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "packages": {
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "opencv-python": "4.13.0.92",
    "Pillow": "11.3.0",
    "matplotlib": "3.10.0",
    "torch": "2.11.0+cpu",
    "torchvision": "0.26.0+cpu"
  },
  "recorded_utc": "2026-06-06T15:16:37.075155+00:00"
}


## 7. Methods text seed

This text will be reused later in manuscript notebooks.

In [7]:
methods_seed = """
The analytical unit was one thermographic image corresponding to one unique horse. The clinical class label was derived from the predefined folder structure: images stored in healthy folders were assigned to the healthy class, whereas images stored in injured folders were assigned to the pathological class. Expert CVAT XML annotations were used only for hotspot localization analyses and not as the primary source of the image-level clinical label. Images without expert hotspot objects were interpreted as no expert-marked hotspot. Any image assigned to the healthy class but containing expert hotspot annotations was flagged for manual review and was not silently relabeled.
""".strip()

(CONFIG_DIR / "methods_text_seed.txt").write_text(methods_seed, encoding="utf-8")
(REPORTS_DIR / "methods_text_seed.txt").write_text(methods_seed, encoding="utf-8")
print(methods_seed)

The analytical unit was one thermographic image corresponding to one unique horse. The clinical class label was derived from the predefined folder structure: images stored in healthy folders were assigned to the healthy class, whereas images stored in injured folders were assigned to the pathological class. Expert CVAT XML annotations were used only for hotspot localization analyses and not as the primary source of the image-level clinical label. Images without expert hotspot objects were interpreted as no expert-marked hotspot. Any image assigned to the healthy class but containing expert hotspot annotations was flagged for manual review and was not silently relabeled.
